<a href="https://colab.research.google.com/github/gemhunter2709/EE604/blob/main/Another_copy_of_oo_laaa_laaa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Navigate to base directory and clean up
%cd /content
!rm -rf ERRNet

# Clone the repository
!git clone https://github.com/Vandermode/ERRNet.git
%cd /content/ERRNet

# Install all dependencies
print("📦 Installing dependencies...")
!pip install -q torch torchvision
!pip install -q scikit-image opencv-python tensorboardX visdom dominate
!pip install -q "Pillow<12.0,>=10.0"

print("✅ Setup complete!")

/content
Cloning into 'ERRNet'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 112 (delta 3), reused 1 (delta 0), pack-reused 103 (from 1)
Receiving objects: 100% (112/112), 980.54 KiB | 3.14 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/ERRNet
📦 Installing dependencies...
✅ Setup complete!


In [ ]:
%cd /content/ERRNet

print("🔧 Applying compatibility patches...")

# Fix scikit-image imports
!sed -i 's/from skimage.measure import compare_ssim, compare_psnr/from skimage.metrics import structural_similarity as compare_ssim, peak_signal_noise_ratio as compare_psnr/g' util/index.py
!sed -i 's/from skimage.measure import compare_ssim/from skimage.metrics import structural_similarity as compare_ssim/g' util/util.py

# Fix Pillow import
!sed -i 's/from PIL import Image, ImageOps, ImageEnhance, PILLOW_VERSION/from PIL import Image, ImageOps, ImageEnhance, __version__ as PILLOW_VERSION/g' data/transforms.py

# Fix torch._utils import
!sed -i 's/from torch._utils import _accumulate/from itertools import accumulate as _accumulate/g' data/torchdata.py

# Fix newline bug in reflect_dataset.py
!sed -i '179s/fn = self.fns\[index\]/fn = self.fns\[index\].strip()/' data/reflect_dataset.py

print("✅ All patches applied successfully!")

/content/ERRNet
🔧 Applying compatibility patches...
✅ All patches applied successfully!


In [ ]:
%cd /content/ERRNet

print("🔧 Fixing util/util.py for Colab...")

# Read the file
with open('util/util.py', 'r') as f:
    content = f.read()

# Replace the problematic code
old_code = "_, term_width = os.popen('stty size', 'r').read().split()\nterm_width = int(term_width)"
new_code = """try:
    _, term_width = os.popen('stty size', 'r').read().split()
    term_width = int(term_width)
except:
    term_width = 80"""

content = content.replace(old_code, new_code)

# Also try with double quotes version
old_code2 = '_, term_width = os.popen("stty size", "r").read().split()\nterm_width = int(term_width)'
content = content.replace(old_code2, new_code)

# Write back
with open('util/util.py', 'w') as f:
    f.write(content)

print("✅ Fixed util/util.py!")

/content/ERRNet
🔧 Fixing util/util.py for Colab...
✅ Fixed util/util.py!


In [ ]:
%cd /content/ERRNet

print("🔍 PART 1: Investigating networks module...\n")

import sys
sys.path.insert(0, '/content/ERRNet')

# Clear any cached imports
for mod in list(sys.modules.keys()):
    if mod.startswith('models') or mod.startswith('util'):
        del sys.modules[mod]

from models import networks

# List all available classes/functions
print("Available in networks module:")
print("="*60)
attrs = [attr for attr in dir(networks) if not attr.startswith('_')]
for attr in attrs:
    obj = getattr(networks, attr)
    if isinstance(obj, type):  # It's a class
        print(f"  ✓ CLASS: {attr}")
    elif callable(obj):  # It's a function
        print(f"  • function: {attr}")

print("\n" + "="*60)
print("\n🔍 PART 2: Inspecting checkpoint file...\n")

import torch

model_path = '/content/ERRNet/checkpoints/errnet/errnet_060_00463920.pt'

# Check if model exists first
import os
if not os.path.exists(model_path):
    print("⚠️  Model file not found yet!")
    print(f"   Please upload to: {model_path}")
else:
    checkpoint = torch.load(model_path, map_location='cpu')

    print(f"Checkpoint type: {type(checkpoint)}")
    print("\nCheckpoint structure:")
    if isinstance(checkpoint, dict):
        for key in checkpoint.keys():
            print(f"  • {key}: {type(checkpoint[key])}")
            if key in ['state_dict', 'model', 'netG'] and isinstance(checkpoint[key], dict):
                print(f"    → Contains {len(checkpoint[key])} weight tensors")
                weight_keys = list(checkpoint[key].keys())[:5]
                print(f"    → First 5 keys:")
                for wk in weight_keys:
                    print(f"       - {wk}")
    else:
        print("Checkpoint is directly a state dict")
        weight_keys = list(checkpoint.keys())[:10]
        print(f"\nFirst 10 weight keys:")
        for wk in weight_keys:
            print(f"  - {wk}")

print("\n" + "="*60)
print("📋 Next: Check the output above and share it!")

/content/ERRNet
🔍 PART 1: Investigating networks module...

Available in networks module:
  ✓ CLASS: Discriminator_VGG
  ✓ CLASS: NLayerDiscriminator
  ✓ CLASS: OrderedDict
  ✓ CLASS: Vgg16
  ✓ CLASS: Vgg19
  • function: debug_network
  • function: define_D
  • function: get_norm_layer
  • function: init_weights
  • function: print_network
  • function: receptive_field
  • function: weights_init_kaiming
  • function: weights_init_normal
  • function: weights_init_orthogonal
  • function: weights_init_xavier


🔍 PART 2: Inspecting checkpoint file...

⚠️  Model file not found yet!
   Please upload to: /content/ERRNet/checkpoints/errnet/errnet_060_00463920.pt

📋 Next: Check the output above and share it!


In [ ]:
%cd /content/ERRNet

# Create necessary directories
!mkdir -p checkpoints/errnet/
!mkdir -p data/real/
!mkdir -p output_results/

print("✅ Directory structure created!")
print("\n" + "="*60)
print("📋 NEXT STEPS:")
print("="*60)
print("1. Upload your MODEL file to: /content/ERRNet/checkpoints/errnet/")
print("   → File name should be: errnet_060_00463920.pt")
print("\n2. Upload your TEST IMAGE(S) to: /content/ERRNet/data/real/")
print("   → Supported formats: .jpg, .jpeg, .png")
print("\n3. After uploading, run the next cell!")
print("="*60)

/content/ERRNet
✅ Directory structure created!

📋 NEXT STEPS:
1. Upload your MODEL file to: /content/ERRNet/checkpoints/errnet/
   → File name should be: errnet_060_00463920.pt

2. Upload your TEST IMAGE(S) to: /content/ERRNet/data/real/
   → Supported formats: .jpg, .jpeg, .png

3. After uploading, run the next cell!


# Make sure the upload is complete in the respective directories before continuing from here!!!


[Get the errnet.pt file from here](https://onedrive.live.com/?id=6234BD5AF87E5DA7%211017&resid=6234BD5AF87E5DA7%211017&ithint=folder&migratedtospo=true&redeem=aHR0cHM6Ly8xZHJ2Lm1zL2YvcyFBcWRkZnZoYXZUUmloM24zVzBQMjljeFZJbGZN&cid=6234bd5af87e5da7&v=validatepermission)

In [ ]:
# PROBLEM


%%writefile /content/ERRNet/universal_inference.py
"""
ERRNet Universal Inference - Works on CPU and GPU
"""
import os
import sys
sys.path.insert(0, '/content/ERRNet')

import torch
from PIL import Image
import torchvision.transforms as transforms
from os.path import join
import glob

def main():
    print("\n" + "="*60)
    print("  🎯 ERRNet - Universal Inference")
    print("="*60)

    # Check device availability
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\n🖥️  Device: {device.upper()}")

    if device == 'cpu':
        print("   ⚠️  Running on CPU (will be slower)")
        print("   💡 Tip: In Colab, enable GPU via Runtime > Change runtime type > T4 GPU")

    model_path = './checkpoints/errnet/errnet_060_00463920.pt'

    if not os.path.exists(model_path):
        print(f"\n❌ Model not found: {model_path}")
        return

    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"   Model size: {model_size_mb:.1f} MB")

    # Import model class
    from models.errnet_model import ERRNetModel

    # Create minimal options object with all required attributes
    class MinimalOptions:
        def __init__(self):
            # Paths
            self.name = 'errnet'
            self.checkpoints_dir = './checkpoints'
            self.icnn_path = model_path

            # Model settings
            self.model = 'errnet'
            self.hyper = True
            self.resume = True
            self.resume_epoch = None
            self.inet = 'errnet'

            # Architecture
            self.ngf = 64
            self.skip = 0
            self.input_nc = 3
            self.output_nc = 3
            self.norm = 'batch'
            self.use_dropout = False
            self.init_type = 'orthogonal'
            self.init_gain = 0.02

            # Device settings - CRITICAL for CPU/GPU compatibility
            self.gpu_ids = [0] if device == 'cuda' else []
            self.isTrain = False

            # Loss settings
            self.lambda_gan = 0
            self.unaligned_loss = 'vgg'
            self.aligned_loss = 'l1'
            self.vgg_layer = '1_1'

            # Display
            self.verbose = False
            self.no_verbose = True
            self.display_id = -1
            self.suffix = ''

    opt = MinimalOptions()

    # Load model
    print("\n📂 Loading model...")
    try:
        model = ERRNetModel()
        model.initialize(opt)
        model.netG.eval()
        print("✅ Model loaded successfully!")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        import traceback
        traceback.print_exc()

        # Try to provide helpful error message
        if 'inet' in str(e):
            print("\n💡 Try checking models/arch/ directory for available architectures")
        return

    # Find images
    input_dir = './data/real/'
    output_dir = './output_results/'
    os.makedirs(output_dir, exist_ok=True)

    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        image_files.extend(glob.glob(join(input_dir, ext)))

    if not image_files:
        print(f"\n❌ No images found in {input_dir}")
        print(f"   Please upload images to: {os.path.abspath(input_dir)}")
        return

    print(f"\n📁 Found {len(image_files)} image(s) to process")

    # Prepare transform
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    # Process images
    success_count = 0

    for img_path in image_files:
        try:
            print(f"\n{'='*60}")
            print(f"🖼️  Processing: {os.path.basename(img_path)}")

            # Load image
            img = Image.open(img_path).convert('RGB')
            w, h = img.size
            print(f"   Original size: {w} x {h} pixels")

            # Pad to multiple of 4 (required by model)
            new_w = ((w + 3) // 4) * 4
            new_h = ((h + 3) // 4) * 4

            if new_w != w or new_h != h:
                padded = Image.new('RGB', (new_w, new_h), (0, 0, 0))
                padded.paste(img, (0, 0))
                img = padded
                print(f"   Padded to: {new_w} x {new_h} pixels")

            # Transform
            img_tensor = transform(img).unsqueeze(0)

            # Move to device (GPU or CPU)
            img_tensor = img_tensor.to(device)

            # Run inference
            print(f"   Running inference...")
            with torch.no_grad():
                output = model.netG(img_tensor)

                # Handle tuple output
                if isinstance(output, tuple):
                    output = output[0]

            # Post-process
            output = output.squeeze(0).cpu()
            output = output * 0.5 + 0.5  # Denormalize from [-1,1] to [0,1]
            output = torch.clamp(output, 0, 1)

            # Convert to PIL Image
            output_img = transforms.ToPILImage()(output)

            # Crop back to original size if we padded
            if new_w != w or new_h != h:
                output_img = output_img.crop((0, 0, w, h))

            # Save
            filename = os.path.basename(img_path)
            name, ext = os.path.splitext(filename)
            output_path = join(output_dir, f"{name}_no_reflection{ext}")
            output_img.save(output_path, quality=95)
            print(f"   ✅ Saved: {os.path.basename(output_path)}")

            success_count += 1

        except Exception as e:
            print(f"\n❌ ERROR processing {os.path.basename(img_path)}:")
            print(f"   {str(e)}")
            import traceback
            traceback.print_exc()

    # Summary
    print(f"\n{'='*60}")
    print(f"  ✅ PROCESSING COMPLETE")
    print("="*60)
    print(f"   Successful: {success_count}/{len(image_files)} images")
    print(f"   📂 Output directory: {output_dir}")
    print("="*60 + "\n")

    # List output files
    output_files = sorted(glob.glob(join(output_dir, '*')))
    if output_files:
        print("📄 Output files:")
        for f in output_files:
            size_kb = os.path.getsize(f) / 1024
            print(f"   • {os.path.basename(f)} ({size_kb:.1f} KB)")

if __name__ == '__main__':
    main()

print("✅ Universal inference script created!")

Writing /content/ERRNet/universal_inference.py


In [ ]:
%cd /content/ERRNet
!python universal_inference.py

/content/ERRNet

  🎯 ERRNet - Universal Inference

🖥️  Device: CUDA

❌ Model not found: ./checkpoints/errnet/errnet_060_00463920.pt
✅ Universal inference script created!


In [ ]:
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt

input_dir = '/content/ERRNet/data/real/'
output_dir = '/content/ERRNet/output_results/'

input_images = sorted(glob.glob(os.path.join(input_dir, '*.*')))
output_images = sorted(glob.glob(os.path.join(output_dir, '*.*')))

print(f"📊 Results: {len(input_images)} input, {len(output_images)} output\n")
print("="*60)

for inp_path in input_images:
    filename = os.path.basename(inp_path)
    name, ext = os.path.splitext(filename)
    out_path = os.path.join(output_dir, f"{name}_no_reflection{ext}")

    if os.path.exists(out_path):
        fig, axes = plt.subplots(1, 2, figsize=(18, 9))

        inp_img = Image.open(inp_path)
        axes[0].imshow(inp_img)
        axes[0].set_title(f'INPUT: {filename}', fontsize=16, fontweight='bold')
        axes[0].axis('off')

        out_img = Image.open(out_path)
        axes[1].imshow(out_img)
        axes[1].set_title('OUTPUT: Reflection Removed', fontsize=16, fontweight='bold', color='green')
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

        print(f"✅ {filename} → {os.path.basename(out_path)}")
        print("="*60 + "\n")
    else:
        print(f"⚠️  No output for: {filename}\n")

📊 Results: 0 input, 0 output

